## Generate data

In [ ]:
import numpy as np
import xarray as xra

NPIXELS_Y = 100
NPIXELS_X = 100
MIN_X = -180
MAX_X = 180
MIN_Y = -90
MAX_Y = 90

y_coords = np.linspace(MIN_Y, MAX_Y, NPIXELS_Y)
x_coords = np.linspace(MIN_X, MAX_X, NPIXELS_X)
x_grid, y_grid = np.meshgrid(x_coords, y_coords)

data = ((x_grid - x_grid.min()) + (y_grid - y_grid.min())) / 2
data = data / data.max()  # Normalize to 0-1

da = xra.DataArray(
    data,
    dims=["y", "x"],
    coords={
        "y": np.linspace(MIN_Y, MAX_Y, NPIXELS_Y),
        "x": np.linspace(MIN_X, MAX_X, NPIXELS_X),
    },
)
da = da.rio.write_crs("EPSG:4326")
da

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

map_proj = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection=map_proj)

da.plot.pcolormesh(ax=ax, transform=map_proj)

ax.coastlines()
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
ax.set_extent([-180, 180, -90, 90], crs=map_proj)  # Optional: set specific map extent

plt.show()

## Xpublish needs extra massaging to geolocate the data

### Reverse y-coordinates to conventional order

Is this needed? Or just the transform? I think without this the data is upside downsies.

In [ ]:
da = da.reindex(y=da.y[::-1])
da

### Add a geotransform

In [ ]:
from affine import Affine

# Calculate pixel size
x_res = (MAX_X - MIN_X) / NPIXELS_X
y_res = (MAX_Y - MIN_Y) / NPIXELS_Y

# transform = Affine.translation(MIN_X, MAX_Y) * Affine.scale(x_res, -y_res)  # noqa: ERA001
transform = Affine.translation(MIN_X - x_res / 2, MAX_Y + y_res / 2) * Affine.scale(
    x_res, -y_res
)

da = da.rio.write_transform(transform)

## Render

This still renders all weird with xpublish. Some tiles don't display depending on zoom level. Wat.

<https://github.com/earth-mover/xpublish-tiles/issues/206>

In [ ]:
from jupyter_xarray_tiler.xpublish import add_data_array

url = await add_data_array(da, rescale=(0, 1))

In [ ]:
url

In [ ]:
import leafmap

m = leafmap.Map(center=[41.321482, -72.932739], zoom=10)
m

In [ ]:
m.add_tile_layer(
    url=url,
    name="Mock data",
    attribution="¯\\_(ツ)_/¯",
)

In [ ]:
da.spatial_ref